# ЛР2 - практика B: самостоятельная работа с транзакциями

**Тема:** несколько таблиц, масштаб, `merge`, `groupby`, `pivot_table`, `qcut` и feature table  
**Формат:** самостоятельная часть второй практики ЛР2  
**Контроль:** одна общая защита после A+B

Вы работаете с четырьмя связанными таблицами и 100 000 транзакций. Цель - не «применить как можно больше методов pandas», а получить корректную таблицу уровня клиента и уметь доказать, что при объединениях не потерялись и не размножились данные.

Данные: `../Task/data`. Результат: `artifacts/client_features.csv`. Ядро: `D:/neural networks/.venv/Scripts/python.exe`.


## 0. Правила работы

- Сначала словами укажите, что означает одна строка каждой таблицы.
- Перед каждым `merge` запишите ожидаемый тип связи.
- Используйте `validate` там, где ожидание можно проверить автоматически.
- После важных трансформаций проверяйте число строк и уникальность ключа.
- Не заполняйте любой `NaN` нулём автоматически: сначала объясните его смысл.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
DATA_DIR = Path("../Task/data")
print("DATA_DIR:", DATA_DIR.resolve())


DATA_DIR: D:\neural networks\Lab 2.2\Task\data


# 1. Загрузка и смысл таблиц

Загрузите:

- `clients.csv` с `parse_dates=["registration_date"]`;
- `transactions.csv` с `parse_dates=["transaction_time"]`;
- `mcc_codes.csv`;
- `transaction_types.csv`.

Выведите `shape`, первые строки и типы столбцов. Затем одной фразой зафиксируйте смысл строки каждой таблицы.

**Ответ:**

Строка clients описывает клиента, transactions описывает операцию. Строка mcc_codes связывает MCC с категорией, transaction_types связывает тип с названием и направлением.


In [2]:
# Загрузка четырёх таблиц и диагностика.
clients = pd.read_csv(DATA_DIR / "clients.csv", parse_dates=["registration_date"])
transactions = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["transaction_time"])
mcc_codes = pd.read_csv(DATA_DIR / "mcc_codes.csv")
transaction_types = pd.read_csv(DATA_DIR / "transaction_types.csv")

for name, table in {
    "clients": clients, "transactions": transactions,
    "mcc_codes": mcc_codes, "transaction_types": transaction_types,
}.items():
    print(name, table.shape)
    display(table.head())
    display(table.dtypes.to_frame("dtype"))

assert len(transactions) == 100_000


clients (5000, 5)


,client_id,age,city,segment,registration_date
0,100001,23,Moscow,regular,2024-06-21
1,100002,62,Yekaterinburg,regular,2023-06-02
2,100003,55,Tomsk,new,2021-06-28
3,100004,43,Samara,regular,2019-06-26
4,100005,42,Saint Petersburg,regular,2019-12-01


,dtype
client_id,int64
age,int64
city,str
segment,str
registration_date,datetime64[us]


transactions (100000, 7)


,transaction_id,client_id,transaction_time,amount,mcc_code,transaction_type_id,channel
0,1,103181,2025-01-01 00:00:00,731.55,4899,100,online
1,2,104681,2025-01-01 00:04:00,67324.54,4829,500,bank_transfer
2,3,103560,2025-01-01 00:08:00,1183.43,5942,100,online
3,4,103226,2025-01-01 00:12:00,266.33,5311,100,card
4,5,100097,2025-01-01 00:20:00,20562.27,4829,300,online


...

,mcc_code,category
0,5411,Groceries
1,5812,Restaurants
2,5912,Pharmacies
3,5541,Fuel
4,5732,Electronics


,dtype
mcc_code,int64
category,str


transaction_types (6, 3)


,transaction_type_id,transaction_type,direction
0,100,Card purchase,expense
1,200,Cash withdrawal,expense
2,300,Transfer out,expense
3,400,Transfer in,income
4,500,Salary,income


,dtype
transaction_type_id,int64
transaction_type,str
direction,str


# 2. Задание 1 - аудит ключей и масштаба

Составьте функцию `key_report(df, key)`, которая возвращает:

- число строк;
- число уникальных ключей;
- число строк, входящих в повторяющиеся значения ключа.

Примените её к ключам:

- `clients.client_id`;
- `transactions.transaction_id`;
- `transactions.client_id`;
- `mcc_codes.mcc_code`;
- `transaction_types.transaction_type_id`.

После таблицы напишите, где повтор ключа является ошибкой, а где - ожидаемым свойством данных.

**Ответ:**

В transactions повтор client_id ожидаем: у клиента несколько операций. Повтор первичного ключа клиента, операции или справочника является ошибкой.


In [3]:
# Уникальность ключей и число строк с повторяющимся ключом.
def key_report(df, key):
    return pd.Series({
        "rows": len(df),
        "unique_keys": df[key].nunique(),
        "duplicated_key_rows": df[key].duplicated(keep=False).sum(),
    })


key_summary = pd.DataFrame({
    "clients.client_id": key_report(clients, "client_id"),
    "transactions.transaction_id": key_report(transactions, "transaction_id"),
    "transactions.client_id": key_report(transactions, "client_id"),
    "mcc_codes.mcc_code": key_report(mcc_codes, "mcc_code"),
    "transaction_types.transaction_type_id": key_report(transaction_types, "transaction_type_id"),
}).T
display(key_summary)

for table, key in [(clients, "client_id"), (transactions, "transaction_id"),
                   (mcc_codes, "mcc_code"), (transaction_types, "transaction_type_id")]:
    assert table[key].notna().all() and table[key].is_unique
assert transactions["client_id"].notna().all()
assert transactions["client_id"].isin(clients["client_id"]).all()


,rows,unique_keys,duplicated_key_rows
clients.client_id,5000,5000,0
transactions.transaction_id,100000,100000,0
transactions.client_id,100000,4600,99975
mcc_codes.mcc_code,14,14,0
transaction_types.transaction_type_id,6,6,0


# 3. Задание 2 - обогащение транзакций через безопасный `merge`

Соберите таблицу `tx`, в которой каждая строка всё ещё означает **одну транзакцию**, но дополнительно есть:

- `category` из MCC-справочника;
- `transaction_type` и `direction` из справочника типов;
- `city` и `segment` клиента.

Требования:

1. везде использовать `how="left"`;
2. указать корректный `validate`;
3. при merge MCC включить `indicator` и посчитать `left_only`;
4. показать несколько неприсоединившихся MCC;
5. заменить отсутствующую `category` на строку `"Unknown"`;
6. проверить, что число строк `tx` равно числу исходных транзакций.

**Ответ:**

Связи many_to_one: ключ справа уникален. Строка tx остаётся одной операцией. Неизвестные MCC сохраняются с category=Unknown.


In [4]:
# Связь many_to_one: много транзакций на один MCC-код.
tx = transactions.merge(
    mcc_codes, on="mcc_code", how="left", validate="many_to_one", indicator="mcc_match",
)
assert len(tx) == len(transactions)
print("MCC merge:")
display(tx["mcc_match"].value_counts().to_frame("rows"))
unmatched_mcc = tx.loc[tx["mcc_match"].eq("left_only"), ["transaction_id", "mcc_code"]]
display(unmatched_mcc.head())
tx["category"] = tx["category"].fillna("Unknown")

# Связь many_to_one: много транзакций на один тип.
tx = tx.merge(transaction_types, on="transaction_type_id", how="left", validate="many_to_one")
assert len(tx) == len(transactions)

# Связь many_to_one: много транзакций на одного клиента.
tx = tx.merge(clients[["client_id", "city", "segment"]], on="client_id", how="left", validate="many_to_one")
assert len(tx) == len(transactions) and tx["transaction_id"].is_unique
assert tx[["transaction_type", "direction", "city", "segment"]].notna().all().all()
assert tx["direction"].isin(["expense", "income"]).all()
assert tx["amount"].ge(0).all()


MCC merge:


,rows
mcc_match,
both,99600
left_only,400
right_only,0


,transaction_id,mcc_code
30,31,9999
197,198,9999
224,225,9999
332,333,9999
708,709,9999


# 4. Задание 3 - извлекаем признаки времени

Из `transaction_time` создайте:

- `month` - номер месяца;
- `weekday` - название дня недели;
- `hour` - час операции.

Затем найдите:

1. пять часов с наибольшим числом транзакций;
2. по каждому `weekday` - число операций и сумму `amount`.

В 2-3 предложениях объясните, почему `parse_dates` лучше сделать при чтении CSV, а не хранить дату обычной строкой.

**Ответ:**

parse_dates создаёт datetime для работы через .dt. Это позволяет извлекать календарные признаки и корректно сортировать даты.


In [5]:
# Временные признаки, популярные часы и итоги по дням недели.
tx["month"] = tx["transaction_time"].dt.month
tx["weekday"] = tx["transaction_time"].dt.day_name()
tx["hour"] = tx["transaction_time"].dt.hour
tx["transaction_day"] = tx["transaction_time"].dt.normalize()

top_hours = tx.groupby("hour").size().sort_values(ascending=False).head(5)
weekday_summary = tx.groupby("weekday").agg(
    transaction_count=("transaction_id", "size"),
    total_amount=("amount", "sum"),
).reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
display(top_hours.rename("transactions").to_frame())
display(weekday_summary)


,transactions
hour,
7,4279
21,4238
8,4221
12,4218
16,4217


,transaction_count,total_amount
weekday,,
Monday,14423,1.206350e+08
Tuesday,13639,1.151634e+08
Wednesday,14532,1.244124e+08
Thursday,14394,1.177794e+08
Friday,14380,1.234052e+08
Saturday,14228,1.225672e+08
Sunday,14404,1.198190e+08


# 5. Задание 4 - `groupby + agg`: одна строка на клиента

Постройте три агрегированные таблицы.

### `base_features`
- `transaction_count`;
- `active_days` - число уникальных календарных дней с операциями;
- `unique_categories`.

### `expense_features` только для `direction == "expense"`
- `expense_count`;
- `total_expense`;
- `avg_expense`;
- `cash_withdrawal_count` - число операций типа `Cash withdrawal`.

### `income_features` только для `direction == "income"`
- `income_count`;
- `total_income`.

Проверьте, что `client_id` уникален во всех трёх агрегатах.

In [6]:
# Клиентские агрегаты: активность, расходы и доходы.
base_features = tx.groupby("client_id", as_index=False).agg(
    transaction_count=("transaction_id", "size"),
    active_days=("transaction_day", "nunique"),
    unique_categories=("category", "nunique"),
)
tx_expense = tx.loc[tx["direction"].eq("expense")].copy()
tx_expense["is_cash_withdrawal"] = tx_expense["transaction_type"].eq("Cash withdrawal")
expense_features = tx_expense.groupby("client_id", as_index=False).agg(
    expense_count=("transaction_id", "size"),
    total_expense=("amount", "sum"),
    avg_expense=("amount", "mean"),
    cash_withdrawal_count=("is_cash_withdrawal", "sum"),
)
tx_income = tx.loc[tx["direction"].eq("income")]
income_features = tx_income.groupby("client_id", as_index=False).agg(
    income_count=("transaction_id", "size"),
    total_income=("amount", "sum"),
)
for table in [base_features, expense_features, income_features]:
    assert table["client_id"].is_unique
    display(table.head())


,client_id,transaction_count,active_days,unique_categories
0,100001,14,11,9
1,100002,13,13,9
2,100003,17,16,9
3,100004,36,33,13
4,100005,25,23,9


,client_id,expense_count,total_expense,avg_expense,cash_withdrawal_count
0,100001,10,26732.01,2673.201000,1
1,100002,13,37776.14,2905.856923,2
2,100003,16,37550.42,2346.901250,2
3,100004,29,95000.57,3275.881724,4
4,100005,20,51690.06,2584.503000,3


,client_id,income_count,total_income
0,100001,4,227942.91
1,100003,1,2248.24
2,100004,7,235445.83
3,100005,5,119314.81
4,100006,1,1574.49


# 6. Задание 5 - `pivot_table`: расходы по категориям

Для расходных операций постройте `pivot_table`:

- строки - `client_id`;
- столбцы - `category`;
- значения - сумма `amount`;
- пропуски заполнить нулём.

Переименуйте получившиеся столбцы в формат `spend_<category>`, предварительно приведя название категории к нижнему регистру и заменив пробелы на `_`.

Это уже не «отчётная сводная таблица», а набор признаков, который затем можно присоединить к клиенту.

In [7]:
# Сумма расходов клиента в каждой категории.
category_spend = tx_expense.pivot_table(
    index="client_id", columns="category", values="amount", aggfunc="sum", fill_value=0,
)
category_spend.columns = [f"spend_{name.lower().replace(' ', '_')}" for name in category_spend.columns]
category_spend = category_spend.rename_axis(columns=None).reset_index()
assert category_spend["client_id"].is_unique and category_spend.columns.is_unique
display(category_spend.head())


,client_id,spend_books,spend_clothing,spend_department_stores,spend_digital_services,spend_electronics,spend_financial_services,spend_fitness,spend_fuel,spend_groceries,spend_money_transfers,spend_pharmacies,spend_restaurants,spend_transport,spend_unknown,spend_utilities
0,100001,0.00,5563.70,523.33,2204.45,364.92,6966.12,0.00,523.23,686.28,1682.41,8217.57,0.00,0.00,0.0,0.00
1,100002,605.66,2974.30,0.00,0.00,5092.89,10663.37,0.00,0.00,3571.82,4700.55,0.00,5247.30,2915.08,0.0,2005.17
2,100003,0.00,551.00,1150.39,4516.57,0.00,5173.42,0.00,0.00,10303.85,13426.02,490.60,1000.02,0.00,0.0,938.55
3,100004,0.00,1798.02,836.80,4499.44,3491.87,22690.32,1011.82,2165.26,1170.18,47617.44,1194.65,5438.78,1243.61,0.0,1842.38
4,100005,0.00,0.00,6612.54,1596.60,2813.86,12646.64,0.00,621.28,5809.97,16784.57,0.00,0.00,4804.60,0.0,0.00


# 7. Задание 6 - собираем итоговую `features`

Начните с **полного** `clients`, затем последовательно добавьте:

- `base_features`;
- `expense_features`;
- `income_features`;
- `category_spend`.

Используйте `how="left"` и `validate="one_to_one"`.

После merge:

- счётчики и суммы для клиентов без операций заполните нулями;
- `avg_expense` **не заполняйте нулём автоматически** - оставьте `NaN` и в комментарии объясните почему;
- проверьте, сколько клиентов вообще не имеет транзакций.

**Ответ:**

Без операций счётчики и суммы равны 0. Среднее без расходов не определено, поэтому avg_expense остаётся NaN. Left join сохраняет всех клиентов.


In [8]:
# Связь one_to_one: клиент и его агрегаты активности.
features = clients.merge(base_features, on="client_id", how="left", validate="one_to_one")
# Связь one_to_one: клиент и его агрегаты расходов.
features = features.merge(expense_features, on="client_id", how="left", validate="one_to_one")
# Связь one_to_one: клиент и его агрегаты доходов.
features = features.merge(income_features, on="client_id", how="left", validate="one_to_one")
# Связь one_to_one: клиент и расходы по категориям.
features = features.merge(category_spend, on="client_id", how="left", validate="one_to_one")

# Нулевые счётчики и суммы означают отсутствие соответствующих операций.
count_columns = ["transaction_count", "active_days", "unique_categories",
                 "expense_count", "cash_withdrawal_count", "income_count"]
spend_columns = category_spend.columns.drop("client_id").tolist()
zero_columns = count_columns + ["total_expense", "total_income"] + spend_columns
features[zero_columns] = features[zero_columns].fillna(0)
features[count_columns] = features[count_columns].astype("int64")
# avg_expense остаётся NaN: без расходов среднее не определено.
no_transactions = features["transaction_count"].eq(0)
print("Клиентов без транзакций:", no_transactions.sum())
assert len(features) == len(clients) and features["client_id"].is_unique
display(features.head())


Клиентов без транзакций: 400


,client_id,age,city,segment,registration_date,transaction_count,active_days,unique_categories,expense_count,total_expense,avg_expense,cash_withdrawal_count,income_count,total_income,spend_books,spend_clothing,spend_department_stores,spend_digital_services,spend_electronics,spend_financial_services,spend_fitness,spend_fuel,spend_groceries,spend_money_transfers,spend_pharmacies,spend_restaurants,spend_transport,spend_unknown,spend_utilities
0,100001,23,Moscow,regular,2024-06-21,14,11,9,10,26732.01,2673.201000,1,4,227942.91,0.00,5563.70,523.33,2204.45,364.92,6966.12,0.00,523.23,686.28,1682.41,8217.57,0.00,0.00,0.0,0.00
1,100002,62,Yekaterinburg,regular,2023-06-02,13,13,9,13,37776.14,2905.856923,2,0,0.00,605.66,2974.30,0.00,0.00,5092.89,10663.37,0.00,0.00,3571.82,4700.55,0.00,5247.30,2915.08,0.0,2005.17
2,100003,55,Tomsk,new,2021-06-28,17,16,9,16,37550.42,2346.901250,2,1,2248.24,0.00,551.00,1150.39,4516.57,0.00,5173.42,0.00,0.00,10303.85,13426.02,490.60,1000.02,0.00,0.0,938.55
3,100004,43,Samara,regular,2019-06-26,36,33,13,29,95000.57,3275.881724,4,7,235445.83,0.00,1798.02,836.80,4499.44,3491.87,22690.32,1011.82,2165.26,1170.18,47617.44,1194.65,5438.78,1243.61,0.0,1842.38
4,100005,42,Saint Petersburg,regular,2019-12-01,25,23,9,20,51690.06,2584.503000,3,5,119314.81,0.00,0.00,6612.54,1596.60,2813.86,12646.64,0.00,621.28,5809.97,16784.57,0.00,0.00,4804.60,0.0,0.00


# 8. Задание 7 - `qcut`: сегментация по расходам

Для клиентов с `total_expense > 0` разбейте `total_expense` на четыре группы одинаковой численности с помощью `pd.qcut`:

`low`, `medium`, `high`, `very_high`.

Добавьте результат в столбец `spend_group`. Для клиентов без расходов оставьте отдельную категорию `no_expense`.

Выведите число клиентов в каждой группе и медиану `total_expense` по группам.

In [9]:
# Квартили положительных расходов и отдельная группа без расходов.
has_expense = features["total_expense"].gt(0)
features["spend_group"] = "no_expense"
features.loc[has_expense, "spend_group"] = pd.qcut(
    features.loc[has_expense, "total_expense"],
    q=4, labels=["low", "medium", "high", "very_high"],
).astype(str)
features["spend_group"] = pd.Categorical(
    features["spend_group"], categories=["no_expense", "low", "medium", "high", "very_high"], ordered=True,
)
spend_summary = features.groupby("spend_group", observed=False).agg(
    clients=("client_id", "size"), median_expense=("total_expense", "median"),
)
display(spend_summary)


,clients,median_expense
spend_group,,
no_expense,409,0.000
low,1148,9907.350
medium,1148,26774.175
high,1147,51628.400
very_high,1148,108456.770


# 9. Задание 8 - sanity checks как часть pipeline

Соберите `pd.Series` минимум из следующих проверок:

- `client_id` уникален в `features`;
- число строк равно `len(clients)`;
- набор клиентов полностью сохранён;
- сумма `transaction_count` равна числу исходных транзакций;
- сумма `expense_count + income_count` равна числу транзакций;
- `total_expense` и `total_income` неотрицательны;
- нет пропусков в `spend_group`.

Все проверки должны дать `True`.

In [10]:
# Сохранение клиентов, операций и денежных сумм.
sanity = pd.Series({
    "unique_client_id": features["client_id"].is_unique,
    "client_count": len(features) == len(clients),
    "all_clients_preserved": set(features["client_id"]) == set(clients["client_id"]),
    "transaction_count": features["transaction_count"].sum() == len(transactions),
    "expense_and_income_count": (features["expense_count"] + features["income_count"]).sum() == len(transactions),
    "nonnegative_totals": features[["total_expense", "total_income"]].ge(0).all().all(),
    "spend_group_complete": features["spend_group"].notna().all(),
    "expense_sum": np.isclose(features["total_expense"].sum(), tx_expense["amount"].sum()),
    "income_sum": np.isclose(features["total_income"].sum(), tx_income["amount"].sum()),
    "category_sum": np.allclose(features[spend_columns].sum(axis=1), features["total_expense"]),
    "undefined_mean": features["avg_expense"].isna().equals(features["expense_count"].eq(0)),
    "per_client_counts": features["transaction_count"].eq(features["expense_count"] + features["income_count"]).all(),
})
display(sanity.to_frame("passed"))
assert sanity.all()


,passed
unique_client_id,True
client_count,True
all_clients_preserved,True
transaction_count,True
expense_and_income_count,True
nonnegative_totals,True
spend_group_complete,True
expense_sum,True
income_sum,True
category_sum,True


# 10. ⭐ Дополнительное задание - одна гипотеза, а не каталог графиков

Выберите **одну** гипотезу и проверьте её агрегатами или одним графиком. Например:

- отличаются ли категории расходов между `premium` и `regular`;
- различается ли доля cash withdrawal по городам;
- отличается ли активность по часам у разных каналов.

До расчёта сформулируйте ожидаемый результат. После расчёта напишите 3-5 предложений: что получилось, что можно утверждать по этим данным и чего утверждать нельзя.

**Ответ:**

Ожидание: медиана avg_expense у premium выше, чем у regular. Сравнение проводится среди клиентов с расходами.


In [11]:
# Проверка одной гипотезы на уровне клиента.
hypothesis_result = features.loc[
    features["segment"].isin(["premium", "regular"]) & features["expense_count"].gt(0)
].groupby("segment").agg(
    clients=("client_id", "size"),
    median_avg_expense=("avg_expense", "median"),
)
display(hypothesis_result.round(2))


,clients,median_avg_expense
segment,,
premium,683,2697.88
regular,2758,2748.92


**Результат гипотезы:** Медианы: premium 2697.88, regular 2748.92. Ожидание не подтвердилось. Учитывались клиенты с расходами, вес каждого одинаков. Значимость и причинность не проверялись; данные синтетические.


# 11. Итог и подготовка к общей защите ЛР2

Ответьте 7-10 предложениями:

1. Что означает «уровень наблюдения» и почему он важен при `merge`?
2. Почему `client_id` может быть уникальным в одной таблице и повторяться в другой?
3. Что проверяет `validate="many_to_one"`?
4. Почему прямой many-to-many merge событий может испортить агрегаты?
5. Зачем анализировать `left_only` после соединения со справочником?
6. Почему клиента без операций разумно сохранить через `left join`?
7. Почему для такого клиента `transaction_count=0`, но `avg_expense=NaN` может быть осмысленнее, чем `0`?
8. Что даёт `qcut`, чего не даёт фиксированная граница вроде `total_expense > 100000`?

На защите важно не только показать готовый DataFrame, но и уметь объяснить, **почему число строк и смысл строки после каждого ключевого шага именно такие**.

**Ответ:**

Уровень наблюдения задаёт смысл строки: клиент или операция. Ключ клиента уникален в clients и повторяется в transactions. many_to_one проверяет уникальность ключа справа. many_to_many размножает строки и может завышать агрегаты. left_only показывает отсутствие соответствия в справочнике. Left join сохраняет клиентов без операций. У них transaction_count=0, а avg_expense не определён. qcut даёт группы близкой численности, фиксированный денежный порог этого не гарантирует.


# 12. Сохранение результата

Итоговая таблица содержит одну строку на клиента. Пустое avg_expense в CSV означает отсутствие расходных операций.

In [12]:
# Экспорт итоговой таблицы признаков.
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
features.to_csv(ARTIFACT_DIR / "client_features.csv", index=False, encoding="utf-8-sig")
print("Сохранено:", ARTIFACT_DIR / "client_features.csv")
print("Размер:", features.shape)

Сохранено: artifacts\client_features.csv
Размер: (5000, 30)


# 13. Источники и проверка

- Материалы: `../Task/student.ipynb`, `../Task/demo.ipynb`, `../Task/data/`.
- ИИ: ChatGPT (Codex), реализация, пояснения и проверка.
- Другие внешние источники не использовались.

- [x] Все задания выполнены, включая дополнительную гипотезу.
- [x] Notebook запускается сверху вниз.
- [x] Ключи и типы связей проверены.
- [x] Клиенты, операции и денежные суммы сохранены.
- [x] Все sanity checks прошли.
- [x] Итоговая таблица сохранена.
- [ ] Готовность объяснить код проверена самостоятельно.
